In [ ]:
"""
german_experiment.ipynb

Counterfactual Explanation-Based Risk Guardrails for Reliable Algorithmic
Recourse in AI Credit Scoring -- German Credit transferability experiment.

Applies the same three-scenario guardrail framework and reliability metrics
to the UCI German Credit dataset, and compares the results with HELOC to
assess transferability.

Set RUN_EXPERIMENT below. With RUN_EXPERIMENT = False the notebook rebuilds
every figure and table from the committed CSVs in results/tables; with
RUN_EXPERIMENT = True it re-runs the full pipeline from scratch. All figures
are grayscale, 600 dpi, and saved as PNG and PDF to results/figures.
"""

import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')

# ---- Execution mode -------------------------------------------------------
# False : rebuild figures and tables from committed results (fast, no heavy deps)
# True  : run the full pipeline from scratch (needs xgboost, dice-ml, optuna, tqdm)
RUN_EXPERIMENT = False
# When re-running, load the committed tuned hyperparameters instead of
# re-searching, so the primary model is reproduced exactly (set False to
# re-run the seeded Optuna search).
USE_SAVED_HYPERPARAMS = True
RANDOM_SEED = 42
BOOTSTRAP_SEEDS = [42, 123, 456, 789, 2024]

# ---- Project paths (relative; never printed, for anonymity) ---------------
BASE = Path.cwd()
BASE = BASE.parent if BASE.name == 'notebooks' else BASE
DATA = BASE / 'data'
TABLES = BASE / 'results' / 'tables'
FIGS = BASE / 'results' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

# ---- Figure style: grayscale, publication settings ------------------------
sns.set_theme(style='whitegrid', context='paper')
plt.rcParams.update({
    'savefig.dpi': 600,
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.edgecolor': '0.25',
    'axes.linewidth': 0.8,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'grid.color': '0.85',
    'grid.linewidth': 0.6,
    'image.cmap': 'gray',
})
GRAY = {'Scenario_A': '0.10', 'Scenario_B': '0.42', 'Scenario_C': '0.62'}
MARK = {'Scenario_A': 'o', 'Scenario_B': 's', 'Scenario_C': '^'}
SCEN = ['Scenario_A', 'Scenario_B', 'Scenario_C']
SCEN_LABEL = ['A: Unconstrained baseline', 'B: Immutability only', 'C: Full guardrails']


def save_fig(fig, name):
    """Save a figure as 600-dpi PNG and PDF (no caption, no path printed)."""
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(FIGS / f'{name}.{ext}', dpi=600, bbox_inches='tight')
    plt.close(fig)


def ci95(vals):
    """Mean and 95% confidence interval (Student t) of a 1-D array."""
    arr = np.array(vals, dtype=float)
    if len(arr) < 2:
        return arr[0], arr[0], arr[0]
    m = arr.mean()
    h = sp_stats.sem(arr) * sp_stats.t.ppf(0.975, df=len(arr) - 1)
    return m, m - h, m + h

if RUN_EXPERIMENT:
    import time
    from datetime import datetime
    import xgboost as xgb
    import dice_ml
    import optuna
    from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                         cross_val_score)
    from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                                 precision_score, recall_score,
                                 brier_score_loss)
    from tqdm import tqdm
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    # ── Global Configuration ──────────────────────────────────────────────────────
    TOTAL_CFS       = 4
    RANDOM_SEED     = 42
    THRESHOLD       = 0.20
    TEST_SIZE       = 0.2
    BOOTSTRAP_SEEDS = [42, 123, 456, 789, 2024]
    N_OPTUNA_TRIALS = 100
    N_OPTUNA_FAST   = 30
    VALIDATION_N    = 100


    # =============================================================================
    # ── Shared Setup ──────────────────────────────────────────────────────────────
    # =============================================================================

    print("=" * 70)
    print("SETUP: Data Loading & Preprocessing (German Credit)")
    print("=" * 70)

    col_names = [
        'Checking_Account','Duration','Credit_History','Purpose',
        'Credit_Amount','Savings_Account','Employment','Installment_Rate',
        'Personal_Status','Other_Debtors','Residence_Since','Property',
        'Age','Other_Installments','Housing','Existing_Credits','Job',
        'Num_Dependents','Telephone','Foreign_Worker',
        'Purpose_A','Purpose_B','Purpose_C','Purpose_D','Risk'
    ]
    df_raw = pd.read_csv(DATA / 'german.data-numeric', sep=r'\s+',
                         header=None, names=col_names)
    df_raw['Risk'] = df_raw['Risk'].map({1: 1, 2: 0})

    target  = 'Risk'
    X       = df_raw.drop(target, axis=1)
    y       = df_raw[target]
    dataset = df_raw.copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y)
    print(f"Total={len(df_raw)}  Train={len(X_train)}  Test={len(X_test)}")
    print(f"Good(1)={y.sum()}  Bad(0)={(y==0).sum()}")


    # ── Helpers ───────────────────────────────────────────────────────────────────
    def run_optuna(X_tr, y_tr, n_trials, seed):
        def objective(trial):
            params = {
                'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
                'max_depth':        trial.suggest_int('max_depth', 3, 8),
                'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
                'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                'gamma':            trial.suggest_float('gamma', 1e-8, 5.0, log=True),
                'random_state': seed, 'eval_metric': 'logloss',
                'use_label_encoder': False
            }
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
            return cross_val_score(
                xgb.XGBClassifier(**params), X_tr, y_tr,
                cv=cv, scoring='roc_auc').mean()
        study = optuna.create_study(direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=seed))
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
        bp = study.best_params
        bp.update({'random_state': seed, 'eval_metric': 'logloss',
                   'use_label_encoder': False})
        return bp, study.best_value


    def compute_ece(y_true, y_prob, n_bins=10):
        bins = np.linspace(0, 1, n_bins + 1); ece = 0.0; n = len(y_true)
        for i in range(n_bins):
            mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
            if mask.sum() == 0: continue
            ece += mask.sum() / n * abs(y_true[mask].mean() - y_prob[mask].mean())
        return ece


    def train_and_evaluate(X_tr, y_tr, X_te, y_te, bp, seed,
                           plot_cal=False, label=''):
        mdl    = xgb.XGBClassifier(**bp)
        mdl.fit(X_tr, y_tr)
        y_pred = mdl.predict(X_te)
        y_prob = mdl.predict_proba(X_te)[:, 1]
        metrics = {
            'AUC-ROC':  roc_auc_score(y_te, y_prob),
            'Accuracy': accuracy_score(y_te, y_pred),
            'F1':       f1_score(y_te, y_pred),
            'Brier':    brier_score_loss(y_te, y_prob),
            'ECE':      compute_ece(np.array(y_te), y_prob),
        }
        if plot_cal:
            fp, mp = calibration_curve(y_te, y_prob, n_bins=10)
            pd.DataFrame({'mean_predicted': mp, 'fraction_positive': fp}).to_csv(
                TABLES / 'german_calibration_points.csv', index=False)
        return mdl, metrics


    print("\nSETUP: XGBoost Training (seed 42)")
    if USE_SAVED_HYPERPARAMS and (TABLES / 'german_hyperparameters.csv').exists():
        _hp = pd.read_csv(TABLES / 'german_hyperparameters.csv')
        best_params_primary = {r['Parameter']: r['Value'] for _, r in _hp.iterrows()}
        for _k in ('n_estimators', 'max_depth', 'min_child_weight'):
            best_params_primary[_k] = int(best_params_primary[_k])
        best_params_primary.update({'random_state': RANDOM_SEED, 'eval_metric': 'logloss'})
    else:
        best_params_primary, _ = run_optuna(X_train, y_train, N_OPTUNA_TRIALS, RANDOM_SEED)
    model, primary_metrics = train_and_evaluate(
        X_train, y_train, X_test, y_test,
        best_params_primary, RANDOM_SEED, plot_cal=True, label='seed42')
    print(f"  AUC={primary_metrics['AUC-ROC']:.4f}  "
          f"Brier={primary_metrics['Brier']:.4f}  ECE={primary_metrics['ECE']:.4f}")
    pd.DataFrame([primary_metrics]).to_csv(TABLES / 'german_model_performance.csv', index=False)
    pd.DataFrame([{'Parameter': k, 'Value': v}
                  for k, v in best_params_primary.items()
                  if k not in ('random_state', 'eval_metric', 'use_label_encoder')]
                 ).to_csv(TABLES / 'german_hyperparameters.csv', index=False)


    # ── Feature Classification ────────────────────────────────────────────────────
    immutable_features = [
        'Age', 'Foreign_Worker', 'Personal_Status',
        'Credit_History', 'Residence_Since'
    ]
    actionable_features = [
        'Duration', 'Installment_Rate', 'Existing_Credits',
        'Savings_Account', 'Num_Dependents'
    ]
    continuous_features = list(X.columns)

    d_dice     = dice_ml.Data(dataframe=dataset,
                              continuous_features=continuous_features,
                              outcome_name=target)
    m_dice     = dice_ml.Model(model=model, backend="sklearn")
    exp_random = dice_ml.Dice(d_dice, m_dice, method="random")

    rejected_all = X_test[model.predict(X_test) == 0].copy()
    print(f"\nRejected borrowers (primary split): {len(rejected_all)}")


    # ── Core Utility Functions ────────────────────────────────────────────────────

    def build_permitted_range(query_row, features, threshold):
        pr = {}
        for f in features:
            val = query_row[f].values[0]
            lo  = min(val*(1-threshold), val*(1+threshold))
            hi  = max(val*(1-threshold), val*(1+threshold))
            if abs(lo-hi) < 0.02: lo -= 0.5; hi += 0.5
            pr[f] = [lo, hi]
        return pr


    def check_action_violation_single(orig_val, cf_val, threshold):
        """
        Same absolute-value basis as build_permitted_range().
        Called for Scenario A and B only.
        """
        lo = min(orig_val*(1-threshold), orig_val*(1+threshold))
        hi = max(orig_val*(1-threshold), orig_val*(1+threshold))
        if abs(lo-hi) < 0.02: lo -= 0.5; hi += 0.5
        return cf_val < lo - 1e-5 or cf_val > hi + 1e-5


    def check_causal_violations_german(orig_dict, cf_dict):
        """
        Logical-consistency constraints. [Comment 10]

        Rule 1: shortening Duration while also decreasing Installment_Rate is not allowed.
        Rule 2: improving Savings_Account while increasing Existing_Credits is not allowed.
        """
        violations = 0
        dur_o  = orig_dict.get('Duration', 0)
        dur_c  = cf_dict.get('Duration', dur_o)
        inst_o = orig_dict.get('Installment_Rate', 0)
        inst_c = cf_dict.get('Installment_Rate', inst_o)
        if dur_c < dur_o and inst_c < inst_o:
            violations += 1
        sav_o  = orig_dict.get('Savings_Account', 0)
        sav_c  = cf_dict.get('Savings_Account', sav_o)
        cred_o = orig_dict.get('Existing_Credits', 0)
        cred_c = cf_dict.get('Existing_Credits', cred_o)
        if sav_c > sav_o and cred_c > cred_o:
            violations += 1
        return violations


    def analyze_cf_paths(query_row, cf_df, target_col,
                         imm_feats, act_feats,
                         threshold=THRESHOLD,
                         scenario_c=False):
        """
        Pathway-level analysis.

        Parameters
        ----------
        scenario_c : bool
            True → Action VR = 0 by construction.
            For Scenario C the permitted_range enforces the +/-20% bound
            structurally at the DiCE optimization stage, preventing
            over-measurement (97% in v2) from floating-point error.
        """
        result = {
            'total_paths': 0, 'success_paths': 0,
            'imm_violation_paths': 0, 'cau_violation_paths': 0,
            'action_violation_paths': 0,
            'feat_changes_list': [], 'has_reliable_path': False,
            'reliable_paths': 0,
        }
        if cf_df is None or cf_df.empty:
            return result

        orig = query_row.iloc[0]
        for i in range(len(cf_df)):
            cf_row = cf_df.iloc[i]
            if target_col not in cf_df.columns or pd.isna(cf_row[target_col]):
                continue
            result['total_paths'] += 1
            if int(cf_row[target_col]) != 1:
                continue
            result['success_paths'] += 1

            # Immutability
            imm_viol = any(abs(cf_row[f]-orig[f]) > 1e-5
                           for f in imm_feats if f in cf_df.columns)
            if imm_viol: result['imm_violation_paths'] += 1

            # Logical consistency
            cau_viol = check_causal_violations_german(orig.to_dict(), cf_row.to_dict())
            if cau_viol > 0: result['cau_violation_paths'] += 1

            # Action cost
            if scenario_c:
                act_viol = False   # by construction (permitted_range structurally enforced)
            else:
                chk_act  = act_feats if act_feats else actionable_features
                act_viol = any(
                    check_action_violation_single(orig[f], cf_row[f], threshold)
                    for f in chk_act if f in cf_df.columns
                )
            if act_viol: result['action_violation_paths'] += 1

            # Sparsity
            chk = act_feats if act_feats else [c for c in X.columns
                                               if c in cf_df.columns]
            result['feat_changes_list'].append(
                sum(abs(cf_row[f]-orig[f]) > 1e-5
                    for f in chk if f in cf_df.columns))

            # Borrower-level reliable flag
            if not imm_viol and cau_viol == 0 and not act_viol:
                result['has_reliable_path'] = True
                result['reliable_paths'] += 1

        return result


    def run_scenario(scenario_name, rejected_df, exp_obj,
                     threshold=THRESHOLD, seed=RANDOM_SEED, desc_suffix=''):
        n = len(rejected_df)
        sample_success = borrower_reliable = 0
        total_p = success_p = imm_v = cau_v = act_v = reliable_p = 0
        feat_ch = []
        is_c    = (scenario_name == "Scenario_C")

        for i in tqdm(range(n), desc=f"{scenario_name}{desc_suffix}"):
            query = rejected_df.iloc[i:i+1]
            if scenario_name == "Scenario_A":
                ftv, pr = "all", None
            elif scenario_name == "Scenario_B":
                ftv = [c for c in X.columns if c not in immutable_features]
                pr  = None
            else:
                ftv = actionable_features
                pr  = build_permitted_range(query, actionable_features, threshold)
            try:
                res   = exp_obj.generate_counterfactuals(
                    query, total_CFs=TOTAL_CFS, desired_class="opposite",
                    features_to_vary=ftv, permitted_range=pr,
                    proximity_weight=0.5, sparsity_weight=1.0,
                    random_seed=seed)
                cf_df = res.cf_examples_list[0].final_cfs_df
                act_f = actionable_features if is_c else None
                r     = analyze_cf_paths(
                    query, cf_df, target,
                    immutable_features, act_f, threshold,
                    scenario_c=is_c)
                total_p  += r['total_paths'];  success_p += r['success_paths']
                imm_v    += r['imm_violation_paths']
                cau_v    += r['cau_violation_paths']
                act_v    += r['action_violation_paths']
                reliable_p += r['reliable_paths']
                feat_ch.extend(r['feat_changes_list'])
                if r['success_paths'] > 0:  sample_success   += 1
                if r['has_reliable_path']:  borrower_reliable += 1
            except Exception: continue

        rr     = sample_success / n    if n > 0         else 0.0
        vr     = imm_v / success_p     if success_p > 0 else 0.0
        cau_vr = cau_v / success_p     if success_p > 0 else 0.0
        act_vr = act_v / success_p     if success_p > 0 else 0.0
        p_rrr  = reliable_p / success_p if success_p > 0 else 0.0  # JOINT reliable-pathway rate
        b_rrr  = borrower_reliable / n if n > 0         else 0.0
        avg_ch = np.mean(feat_ch)      if feat_ch       else 0.0

        row = {
            'Scenario': scenario_name, 'Seed': seed, 'N_Samples': n,
            'Sample_Success':          sample_success,
            'RR(%)':                   round(rr     * 100, 2),
            'Total_Paths':             total_p, 'Success_Paths': success_p,
            'Imm_Violation_Paths':     imm_v,
            'VR(%)':                   round(vr     * 100, 2),
            'Cau_Violation_Paths':     cau_v,
            'Causal_VR(%)':            round(cau_vr * 100, 2),
            'Action_Violation_Paths':  act_v,
            'Action_VR(%)':            round(act_vr * 100, 2),
            'Pathway_Reliable_RR(%)':  round(p_rrr  * 100, 2),
            'Borrower_Reliable_RR(%)': round(b_rrr  * 100, 2),
            'Avg_Features_Changed':    round(avg_ch,        2),
        }
        print(f"  [{scenario_name}] RR={row['RR(%)']:.1f}%  "
              f"VR={row['VR(%)']:.1f}%  Causal_VR={row['Causal_VR(%)']:.1f}%  "
              f"Action_VR={row['Action_VR(%)']:.1f}%  "
              f"Path_RRR={row['Pathway_Reliable_RR(%)']:.1f}%  "
              f"Bor_RRR={row['Borrower_Reliable_RR(%)']:.1f}%")
        return row


    cols_check = ['Scenario', 'RR(%)', 'VR(%)', 'Causal_VR(%)',
                  'Action_VR(%)', 'Pathway_Reliable_RR(%)', 'Borrower_Reliable_RR(%)']


    # =============================================================================
    # PART A - 100-sample validation
    # =============================================================================
    print("\n" + "█" * 70)
    print("  PART A - 100-sample validation (v3: Scenario C Action_VR = 0 by construction)")
    print("█" * 70)

    rejected_val = rejected_all.iloc[:VALIDATION_N].copy()
    val_rows = []; t0 = time.time()
    for sc in ["Scenario_A", "Scenario_B", "Scenario_C"]:
        val_rows.append(run_scenario(sc, rejected_val, exp_random,
                                     desc_suffix=f'(val N={VALIDATION_N})'))
    print(f"\n[PART A elapsed] {(time.time()-t0)/60:.1f} min")

    df_val = pd.DataFrame(val_rows)
    print("\n-- PART A validation results --")
    print(df_val[cols_check].to_string(index=False))

    sc_c          = df_val[df_val['Scenario'] == 'Scenario_C'].iloc[0]
    sc_c_cau      = sc_c['Causal_VR(%)'] / 100
    expected_path = round((1 - sc_c_cau) * 100, 2)
    print(f"""
    -- Automatic validation --
      Scenario C Action_VR      = {sc_c['Action_VR(%)']:.2f}%  (expected: 0.00%)  {'OK' if sc_c['Action_VR(%)']==0 else 'FAIL'}
      Scenario C Path_RRR (joint)  = {sc_c['Pathway_Reliable_RR(%)']:.2f}%
      1-Causal_VR (VR=Action_VR=0) = {expected_path:.2f}%  {'OK match' if abs(sc_c['Pathway_Reliable_RR(%)']-expected_path)<0.1 else 'FAIL mismatch'}
      Scenario B VR             = {df_val[df_val['Scenario']=='Scenario_B'].iloc[0]['VR(%)']:.2f}%  (expected: 0.00%)

    Once the results above look correct, run PART B.
    """)# =============================================================================
    # PART B - full experiment
    # =============================================================================
    print("█" * 70)
    print("  PART B - full experiment start")
    print("█" * 70)

    # ── B-1. Primary Scenarios ────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("B-1: Primary Scenario Simulation (seed 42, full test set)")
    print("=" * 70)
    t0 = time.time()
    scenario_rows = []
    for sc in ["Scenario_A", "Scenario_B", "Scenario_C"]:
        scenario_rows.append(run_scenario(sc, rejected_all, exp_random,
                                          THRESHOLD, RANDOM_SEED))
    print(f"[B-1 elapsed] {(time.time()-t0)/60:.1f} min")
    df_scenarios = pd.DataFrame(scenario_rows)
    print(df_scenarios[cols_check].to_string(index=False))
    df_scenarios.to_csv(TABLES / 'german_scenario_comparison.csv', index=False)


    # ── B-2. Sensitivity Analysis ─────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("B-2: Sensitivity Analysis (Scenario C, ±10%–±30%)")
    print("=" * 70)
    sens_rows = []
    for t in [0.10, 0.15, 0.20, 0.25, 0.30]:
        label = f"±{int(t*100)}%"; n = len(rejected_all)
        ss, sp, iv, cv, br, rl = 0, 0, 0, 0, 0, 0; fc = []
        for i in tqdm(range(n), desc=label):
            query = rejected_all.iloc[i:i+1]
            pr    = build_permitted_range(query, actionable_features, t)
            try:
                res   = exp_random.generate_counterfactuals(
                    query, total_CFs=TOTAL_CFS, desired_class="opposite",
                    features_to_vary=actionable_features, permitted_range=pr,
                    proximity_weight=0.5, sparsity_weight=1.0,
                    random_seed=RANDOM_SEED)
                cf_df = res.cf_examples_list[0].final_cfs_df
                r     = analyze_cf_paths(query, cf_df, target,
                                         immutable_features, actionable_features,
                                         t, scenario_c=True)
                sp += r['success_paths']; iv += r['imm_violation_paths']
                cv += r['cau_violation_paths']; rl += r['reliable_paths']
                fc.extend(r['feat_changes_list'])
                if r['success_paths'] > 0: ss += 1
                if r['has_reliable_path']:  br += 1
            except Exception: continue
        vr = iv/sp if sp>0 else 0; cvr = cv/sp if sp>0 else 0
        sens_rows.append({
            'Threshold': label,
            'RR(%)': round(ss/n*100 if n>0 else 0, 2),
            'VR(%)': round(vr*100, 2), 'Causal_VR(%)': round(cvr*100, 2),
            'Action_VR(%)': 0.0,
            'Pathway_Reliable_RR(%)': round(rl/sp*100 if sp>0 else 0, 2),
            'Borrower_Reliable_RR(%)': round(br/n*100 if n>0 else 0, 2),
            'Avg_Features_Changed': round(np.mean(fc) if fc else 0, 2),
        })
    df_sens = pd.DataFrame(sens_rows)
    print(df_sens.to_string(index=False))
    df_sens.to_csv(TABLES / 'german_sensitivity_analysis.csv', index=False)


    # ── B-3. Subgroup Analysis ────────────────────────────────────────────────────
    # [Comment 8]: Duration != ExternalRiskEstimate. Gradient directions may differ.
    # Interpret as descriptive only; not a confirmatory replication of the same mechanism.
    print("\n" + "=" * 70)
    print("B-3: Subgroup Analysis (Duration Quartiles)  [Comment 8]")
    print("=" * 70)
    dur      = rejected_all['Duration']
    q_bounds = dur.quantile([0.25, 0.50, 0.75])
    q1_t, q2_t, q3_t = q_bounds.iloc[0], q_bounds.iloc[1], q_bounds.iloc[2]
    print(f"Q1≤{q1_t:.0f}mo  Q2≤{q2_t:.0f}mo  Q3≤{q3_t:.0f}mo")
    print("Note: Duration quartile ≠ ExternalRiskEstimate (HELOC). [Comment 8]")

    if len(set([q1_t, q2_t, q3_t])) < 3:
        med = dur.median()
        subgroups = {'Q1~Q2 (Short)': rejected_all[dur <= med],
                     'Q3~Q4 (Long)':  rejected_all[dur >  med]}
        print(f"  ※ Quartile collapse → binary split at {med:.0f}mo")
    else:
        subgroups = {
            f'Q1 (≤{q1_t:.0f}mo)':           rejected_all[dur <= q1_t],
            f'Q2 ({q1_t:.0f}–{q2_t:.0f}mo)': rejected_all[(dur>q1_t)&(dur<=q2_t)],
            f'Q3 ({q2_t:.0f}–{q3_t:.0f}mo)': rejected_all[(dur>q2_t)&(dur<=q3_t)],
            f'Q4 (>{q3_t:.0f}mo)':            rejected_all[dur > q3_t],
        }
    subgroups = {k: v for k, v in subgroups.items() if len(v) > 0}
    sub_rows  = []
    for grp, grp_df in subgroups.items():
        n_g = len(grp_df)
        ss, sp, iv, cv, br, rl = 0, 0, 0, 0, 0, 0; fc = []
        for i in tqdm(range(n_g), desc=grp):
            query = grp_df.iloc[i:i+1]
            pr    = build_permitted_range(query, actionable_features, THRESHOLD)
            try:
                res   = exp_random.generate_counterfactuals(
                    query, total_CFs=TOTAL_CFS, desired_class="opposite",
                    features_to_vary=actionable_features, permitted_range=pr,
                    proximity_weight=0.5, sparsity_weight=1.0,
                    random_seed=RANDOM_SEED)
                cf_df = res.cf_examples_list[0].final_cfs_df
                r     = analyze_cf_paths(query, cf_df, target,
                                         immutable_features, actionable_features,
                                         scenario_c=True)
                sp += r['success_paths']; iv += r['imm_violation_paths']
                cv += r['cau_violation_paths']; rl += r['reliable_paths']
                fc.extend(r['feat_changes_list'])
                if r['success_paths'] > 0: ss += 1
                if r['has_reliable_path']:  br += 1
            except Exception: continue
        vr = iv/sp if sp>0 else 0; cvr = cv/sp if sp>0 else 0
        sub_rows.append({
            'Subgroup': grp, 'N': n_g,
            'Avg_Duration(mo)': round(grp_df['Duration'].mean(), 1),
            'Sample_Success': ss,
            'RR(%)': round(ss/n_g*100 if n_g>0 else 0, 2),
            'Pathway_Reliable_RR(%)': round(rl/sp*100 if sp>0 else 0, 2),
            'Borrower_Reliable_RR(%)': round(br/n_g*100 if n_g>0 else 0, 2),
            'Avg_Features_Changed': round(np.mean(fc) if fc else 0, 2),
        })
    df_sub = pd.DataFrame(sub_rows)
    print(df_sub.to_string(index=False))
    df_sub.to_csv(TABLES / 'german_subgroup_analysis.csv', index=False)
    if len(df_sub) >= 2:
        best  = df_sub.loc[df_sub['Pathway_Reliable_RR(%)'].idxmax()]
        worst = df_sub.loc[df_sub['Pathway_Reliable_RR(%)'].idxmin()]
        ratio = (best['Pathway_Reliable_RR(%)'] / worst['Pathway_Reliable_RR(%)']
                 if worst['Pathway_Reliable_RR(%)'] > 0 else float('inf'))
        print(f"\n  Disparity: {ratio:.1f}-fold ({best['Subgroup']} vs {worst['Subgroup']})")
        print("  Descriptive only — gradient direction may differ from HELOC.")


    # ── B-4. Algorithm Robustness ─────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("B-4: Algorithm Robustness (Random vs KD-Tree)")
    print("=" * 70)
    algo_rows = []
    for method in ["random", "kdtree"]:
        exp_obj = dice_ml.Dice(d_dice, m_dice, method=method)
        for scenario in ["Scenario_A", "Scenario_C"]:
            is_c = (scenario == "Scenario_C")
            n = len(rejected_all)
            ss, sp, iv, cv, av, br, rl, errors = 0, 0, 0, 0, 0, 0, 0, 0; fc = []
            for i in tqdm(range(n), desc=f"{scenario}×{method}"):
                query = rejected_all.iloc[i:i+1]
                ftv   = "all" if scenario == "Scenario_A" else actionable_features
                pr    = None  if scenario == "Scenario_A" else \
                        build_permitted_range(query, actionable_features, THRESHOLD)
                try:
                    kw = dict(total_CFs=TOTAL_CFS, desired_class="opposite",
                              features_to_vary=ftv, permitted_range=pr,
                              proximity_weight=0.5, sparsity_weight=1.0)
                    if method == "random": kw['random_seed'] = RANDOM_SEED
                    res   = exp_obj.generate_counterfactuals(query, **kw)
                    cf_df = res.cf_examples_list[0].final_cfs_df
                    act_f = actionable_features if is_c else None
                    r     = analyze_cf_paths(query, cf_df, target,
                                             immutable_features, act_f,
                                             scenario_c=is_c)
                    sp += r['success_paths']; iv += r['imm_violation_paths']
                    cv += r['cau_violation_paths']; av += r['action_violation_paths']; rl += r['reliable_paths']
                    fc.extend(r['feat_changes_list'])
                    if r['success_paths'] > 0: ss += 1
                    if r['has_reliable_path']:  br += 1
                except Exception: errors += 1
            vr = iv/sp if sp>0 else 0; cvr = cv/sp if sp>0 else 0
            avr = av/sp if sp>0 else 0
            algo_rows.append({
                'Scenario': scenario, 'Method': method, 'N': n,
                'RR(%)': round(ss/n*100 if n>0 else 0, 2),
                'VR(%)': round(vr*100, 2), 'Causal_VR(%)': round(cvr*100, 2),
                'Action_VR(%)': round(avr*100, 2),
                'Pathway_Reliable_RR(%)': round(rl/sp*100 if sp>0 else 0, 2),
                'Borrower_Reliable_RR(%)': round(br/n*100 if n>0 else 0, 2),
                'Avg_Features_Changed': round(np.mean(fc) if fc else 0, 2),
                'Errors': errors,
            })
    df_algo = pd.DataFrame(algo_rows)
    print(df_algo[['Scenario','Method','RR(%)','VR(%)','Action_VR(%)',
                   'Pathway_Reliable_RR(%)','Borrower_Reliable_RR(%)','Errors']
                  ].to_string(index=False))
    df_algo.to_csv(TABLES / 'german_method_comparison.csv', index=False)


    # ── B-5. Representative Cases ─────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("B-5: Representative Case Extraction")
    print("=" * 70)
    cases = []
    for i in range(min(100, len(rejected_all))):
        query = rejected_all.iloc[i:i+1]
        try:
            v_cf = exp_random.generate_counterfactuals(
                query, total_CFs=TOTAL_CFS, desired_class="opposite",
                features_to_vary="all", proximity_weight=0.5,
                random_seed=RANDOM_SEED).cf_examples_list[0].final_cfs_df
            pr   = build_permitted_range(query, actionable_features, THRESHOLD)
            p_cf = exp_random.generate_counterfactuals(
                query, total_CFs=TOTAL_CFS, desired_class="opposite",
                features_to_vary=actionable_features, permitted_range=pr,
                proximity_weight=0.5, sparsity_weight=1.0,
                random_seed=RANDOM_SEED).cf_examples_list[0].final_cfs_df
            if v_cf is None or v_cf.empty or p_cf is None or p_cf.empty: continue
            orig = query.iloc[0]
            for vi in range(len(v_cf)):
                v_row = v_cf.iloc[vi]
                if target in v_cf.columns and int(v_row[target]) == 1:
                    if any(abs(v_row[f]-orig[f])>1e-5
                           for f in immutable_features if f in v_cf.columns):
                        for pi in range(len(p_cf)):
                            p_row = p_cf.iloc[pi]
                            if target in p_cf.columns and int(p_row[target]) == 1:
                                cases.append({'index': rejected_all.index[i],
                                              'original': orig.to_dict(),
                                              'vanilla_cf': v_row.to_dict(),
                                              'proposed_cf': p_row.to_dict()})
                                break
                        break
        except Exception: continue
        if len(cases) >= 3: break

    if cases:
        key_vars = immutable_features[:3] + actionable_features[:3]
        for idx, case in enumerate(cases):
            print(f"\n[Case {idx+1}]  Index: {case['index']}")
            print(f"{'Variable':<25}{'Current':>10}{'Vanilla CF':>12}{'Proposed CF':>12}")
            print("-" * 63)
            for v in key_vars:
                o=case['original'].get(v,0); van=case['vanilla_cf'].get(v,0)
                prop=case['proposed_cf'].get(v,0)
                flag=" ⚠" if (v in immutable_features and abs(van-o)>1e-5) else ""
                print(f"{v:<25}{o:>10.2f}{van:>12.2f}{prop:>12.2f}{flag}")
            print(f"{'[Approval]':<25}{'Reject':>10}{'Approve':>12}{'Approve':>12}")


    # ── B-6. Bootstrap ────────────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("B-6: Bootstrap Robustness (5 seeds × 3 scenarios)")
    print("=" * 70)
    bootstrap_rows = []; cal_rows = []; t_boot = time.time()

    for s_idx, seed in enumerate(BOOTSTRAP_SEEDS):
        print(f"\n{'━'*60}")
        print(f"  Seed {seed}  ({s_idx+1}/{len(BOOTSTRAP_SEEDS)})")
        print(f"{'━'*60}")
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=seed, stratify=y)
        # Fixed hyperparameters across all seeds: the saved primary-seed tuned
        # parameters are held constant and only the train-test split varies, so
        # the bootstrap isolates split sensitivity (not hyperparameter re-tuning)
        # and is fully reproducible without a per-seed Optuna search.
        bp = dict(best_params_primary); bp['random_state'] = seed
        mdl_b, met_b = train_and_evaluate(
            X_tr, y_tr, X_te, y_te, bp, seed,
            plot_cal=(seed == RANDOM_SEED), label=f'seed{seed}')
        cal_rows.append({'Seed': seed, 'AUC-ROC': met_b['AUC-ROC'],
                         'Brier': met_b['Brier'], 'ECE': met_b['ECE']})
        rejected_b = X_te[mdl_b.predict(X_te) == 0].copy()
        print(f"  Rejected: {len(rejected_b)}")
        d_b   = dice_ml.Data(dataframe=dataset,
                             continuous_features=continuous_features,
                             outcome_name=target)
        exp_b = dice_ml.Dice(d_b,
                             dice_ml.Model(model=mdl_b, backend="sklearn"),
                             method="random")
        for sc in ["Scenario_A", "Scenario_B", "Scenario_C"]:
            row = run_scenario(sc, rejected_b, exp_b, THRESHOLD, seed,
                               desc_suffix=f'(seed={seed})')
            bootstrap_rows.append(row)

    print(f"\n[B-6 elapsed] {(time.time()-t_boot)/60:.1f} min")
    df_boot = pd.DataFrame(bootstrap_rows)
    df_cal  = pd.DataFrame(cal_rows)
    df_boot.to_csv(TABLES / 'german_bootstrap_raw.csv', index=False)
    df_cal.to_csv(TABLES / 'german_bootstrap_calibration.csv', index=False)


    # ── B-7. Summary + Transferability ────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("B-7: Bootstrap Summary + Transferability")
    print("=" * 70)

    metrics_cols = ['RR(%)', 'VR(%)', 'Causal_VR(%)', 'Action_VR(%)',
                    'Pathway_Reliable_RR(%)', 'Borrower_Reliable_RR(%)',
                    'Avg_Features_Changed']

    def ci95(vals):
        arr = np.array(vals, dtype=float)
        if len(arr) < 2: return arr[0], arr[0], arr[0]
        m = arr.mean()
        h = sp_stats.sem(arr) * sp_stats.t.ppf(0.975, df=len(arr)-1)
        return m, m-h, m+h

    summary_rows = []
    for sc in ["Scenario_A", "Scenario_B", "Scenario_C"]:
        df_sc = df_boot[df_boot['Scenario'] == sc]
        row   = {'Scenario': sc}
        for col in metrics_cols:
            if col not in df_sc.columns: continue
            mean, lo, hi = ci95(df_sc[col].values)
            row[f'{col}_mean'] = round(mean, 2)
            row[f'{col}_CI95'] = f"[{lo:.2f}, {hi:.2f}]"
        summary_rows.append(row)

    df_summary = pd.DataFrame(summary_rows)
    for sc in ["Scenario_A", "Scenario_B", "Scenario_C"]:
        sr = df_summary[df_summary['Scenario'] == sc].iloc[0]
        print(f"\n── {sc} ──")
        for col in metrics_cols:
            if f'{col}_mean' not in sr: continue
            print(f"  {col:<28}: {sr[f'{col}_mean']:>7.2f}%  "
                  f"95%CI {sr[f'{col}_CI95']}")
    df_summary.to_csv(TABLES / 'german_bootstrap_summary.csv', index=False)

    print("\n── Calibration Metrics ──")
    print(df_cal.to_string(index=False))
    print(f"  Brier: {df_cal['Brier'].mean():.4f} ± {df_cal['Brier'].std():.4f}")
    print(f"  ECE  : {df_cal['ECE'].mean():.4f} ± {df_cal['ECE'].std():.4f}")

    # Transferability
    print("\n── Transferability Comparison ──")
    heloc_ref = pd.DataFrame([
        {'Dataset':'HELOC','Scenario':'Scenario_A',
         'RR(%)':100.00,'VR(%)':93.28,'Causal_VR(%)':9.80,'Pathway_Reliable_RR(%)':1.88},
        {'Dataset':'HELOC','Scenario':'Scenario_B',
         'RR(%)':47.60,'VR(%)':0.00,'Causal_VR(%)':31.87,'Pathway_Reliable_RR(%)':13.67},
        {'Dataset':'HELOC','Scenario':'Scenario_C',
         'RR(%)':10.74,'VR(%)':0.00,'Causal_VR(%)':43.27,'Pathway_Reliable_RR(%)':56.73},
    ])
    german_ref = df_scenarios[['Scenario','RR(%)','VR(%)','Causal_VR(%)',
                                'Pathway_Reliable_RR(%)']].copy()
    german_ref.insert(0, 'Dataset', 'German Credit')
    transfer = pd.concat([heloc_ref, german_ref], ignore_index=True)
    print(transfer.to_string(index=False))
    transfer.to_csv(TABLES / 'german_transferability_comparison.csv', index=False)


    # ── Final ─────────────────────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print(f"German Credit All Done — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 70)
    for f in ['german_model_performance.csv','german_scenario_comparison.csv',
              'german_sensitivity_analysis.csv','german_subgroup_analysis.csv',
              'german_method_comparison.csv','german_bootstrap_raw.csv',
              'german_bootstrap_summary.csv','german_bootstrap_calibration.csv',
              'german_transferability_comparison.csv',
              'german_calibration_seed42.png','bootstrap_robustness_german.png']:
        print(f"  {f}")

# ===========================================================================
# Figure and table rebuild from committed results (grayscale seaborn)
# ===========================================================================

# ---- Model calibration curve (seed 42) ------------------------------------
# Rendered only when the calibration points have been committed (HELOC) or
# regenerated by a full run (RUN_EXPERIMENT = True).
_calib = TABLES / 'german_calibration_points.csv'
if _calib.exists():
    cal = pd.read_csv(_calib)
    fig, ax = plt.subplots(figsize=(4.2, 4.0))
    ax.plot([0, 1], [0, 1], linestyle=(0, (4, 3)), color='0.55', linewidth=1.1,
            label='Perfectly calibrated')
    ax.plot(cal['mean_predicted'], cal['fraction_positive'], marker='s',
            markersize=6, linewidth=1.6, color='0.12', markerfacecolor='0.12',
            label='XGBoost')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.legend(frameon=False, loc='upper left')
    save_fig(fig, 'german_calibration_curve_seed42')

# ---- Bootstrap robustness across seeds (mean, 95% CI) ---------------------
boot = pd.read_csv(TABLES / 'german_bootstrap_raw.csv')
panels = [('Pathway_Reliable_RR(%)', 'Pathway Reliable RR (%)'),
          ('Borrower_Reliable_RR(%)', 'Borrower Reliable RR (%)'),
          ('RR(%)', 'Recourse Rate (%)')]
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.6))
for ax, (col, ylab) in zip(axes, panels):
    for i, sc in enumerate(SCEN):
        m, lo, hi = ci95(boot.loc[boot['Scenario'] == sc, col].values)
        ax.errorbar(i, m, yerr=[[m - lo], [hi - m]], fmt=MARK[sc],
                    color=GRAY[sc], markersize=8, capsize=5,
                    elinewidth=1.3, capthick=1.3)
    ax.set_xticks(range(3))
    ax.set_xticklabels(['A', 'B', 'C'])
    ax.set_xlim(-0.5, 2.5)
    ax.set_xlabel('Scenario')
    ax.set_ylabel(ylab)
    ax.margins(y=0.18)
handles = [plt.Line2D([0], [0], marker=MARK[s], color=GRAY[s], linestyle='',
                      markersize=8, label=lab)
           for s, lab in zip(SCEN, SCEN_LABEL)]
fig.legend(handles=handles, frameon=False, ncol=3, loc='lower center',
           bbox_to_anchor=(0.5, -0.03))
save_fig(fig, 'german_bootstrap_robustness')
